# ControlNet model experiments


In [ ]:
import os, sys, json, ast
import numpy as np
import torch
import matplotlib.pyplot as plt
from functools import partial
from diffusers.models.unets.unet_2d import UNet2DModel
from diffusers.training_utils import EMAModel

from utils import (
    NpyImageDataset, channel_normalize, channel_denormalize,
    get_device, make_plot, make_difference_plot,
)
from controlnet.model import SeaIceControlNet, ControlledUNet
from controlnet.sampler import ControlNetSampler


In [ ]:
CHECKPOINT_DIR = "checkpoints/controlnet"
RUN_NAME = None
PRETRAINED_UNET_PATH_OVERRIDE = None

DATA_ROOT = "/mnt/sciml/a.sadreev/sea_ice_data"
SATELLITE_DATA_PATH = "/mnt/sciml/data_assimilation/sral_si/sral_new_format/"
WATER_MASK_SOURCE = "/mnt/sciml/data_assimilation/da_arctic_2015-2024_v0.1/preds/ocean+atmosphere_24_2015-01-17.npy"

N_SAMPLES = 25
NUM_TIMESTEPS = 50
IMAGE_SIZE = (320, 256)
SATELLITE_FILE_INDEX = 1500

DEVICE = get_device()
print(f"device: {DEVICE}")

with open(os.path.join(DATA_ROOT, "train", "stats.json")) as f:
    stats = json.load(f)
CHANNEL_MEAN = tuple(stats["mean"])
CHANNEL_STD = tuple(stats["std"])
print(f"channel_mean={CHANNEL_MEAN}  channel_std={CHANNEL_STD}")


## Загрузка модели


In [ ]:
if RUN_NAME is None:
    assert os.path.isdir(CHECKPOINT_DIR), f"Не найден каталог {CHECKPOINT_DIR}"
    runs = sorted(d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("run_"))
    assert runs, f"Нет run_* в {CHECKPOINT_DIR}"
    RUN_NAME = runs[-1]

run_dir = os.path.join(CHECKPOINT_DIR, RUN_NAME)
print(f"Используем run: {run_dir}")

cfg_path = os.path.join(run_dir, "config.json")
saved_cfg = {}
if os.path.exists(cfg_path):
    with open(cfg_path) as f:
        saved_cfg = json.load(f)
    print(json.dumps({k: v for k, v in saved_cfg.items() if k not in ("channel_mean", "channel_std")}, indent=2))

def parse_saved_value(value, default=None):
    if value is None:
        return default
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except Exception:
            return value
    return value

IMAGE_SIZE = tuple(parse_saved_value(saved_cfg.get("image_size"), IMAGE_SIZE))
IN_CHANNELS = int(parse_saved_value(saved_cfg.get("in_channels"), 4))
OUT_CHANNELS = int(parse_saved_value(saved_cfg.get("out_channels"), 2))
CONDITIONING_CHANNELS = int(parse_saved_value(saved_cfg.get("controlnet_conditioning_channels"), 3))
PRETRAINED_UNET_PATH = PRETRAINED_UNET_PATH_OVERRIDE or saved_cfg.get("pretrained_unet_path")

assert PRETRAINED_UNET_PATH, "Не задан путь к pretrained EMA UNet"
assert os.path.exists(PRETRAINED_UNET_PATH), f"Файл не найден: {PRETRAINED_UNET_PATH}"

controlnet_ema_path = os.path.join(run_dir, "controlnet_ema_best.pth")
if not os.path.exists(controlnet_ema_path):
    controlnet_ema_path = os.path.join(run_dir, "controlnet_ema_last.pth")
assert os.path.exists(controlnet_ema_path), f"Не найден controlnet EMA checkpoint в {run_dir}"

print(f"pretrained unet: {PRETRAINED_UNET_PATH}")
print(f"controlnet ema: {controlnet_ema_path}")
print(f"image_size={IMAGE_SIZE}, in_channels={IN_CHANNELS}, out_channels={OUT_CHANNELS}, cond_channels={CONDITIONING_CHANNELS}")


In [ ]:
unet = UNet2DModel(
    sample_size=IMAGE_SIZE,
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 512, 512),
    down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D"),
)

base_ema = EMAModel(unet.parameters(), decay=0.999)
try:
    base_ema_state = torch.load(PRETRAINED_UNET_PATH, map_location="cpu", weights_only=True)
except TypeError:
    base_ema_state = torch.load(PRETRAINED_UNET_PATH, map_location="cpu")
base_ema.load_state_dict(base_ema_state)
base_ema.copy_to(unet.parameters())

controlnet = SeaIceControlNet(
    unet,
    conditioning_channels=CONDITIONING_CHANNELS,
)
controlnet_ema = EMAModel(controlnet.parameters(), decay=0.999)
try:
    controlnet_ema_state = torch.load(controlnet_ema_path, map_location="cpu", weights_only=True)
except TypeError:
    controlnet_ema_state = torch.load(controlnet_ema_path, map_location="cpu")
controlnet_ema.load_state_dict(controlnet_ema_state)
controlnet_ema.copy_to(controlnet.parameters())

model = ControlledUNet(unet, controlnet)
model.eval().to(DEVICE)
sampler = ControlNetSampler(model)
print("Модель загружена.")


## Валидационные данные — равномерно расставленные сэмплы


In [ ]:
transform = partial(channel_normalize, channel_mean=CHANNEL_MEAN, channel_std=CHANNEL_STD)

val_dataset = NpyImageDataset(
    folder=os.path.join(DATA_ROOT, "valid"),
    transform=transform,
    preload=False,
    mmap_mode='r',
)
print(f"Всего val-сэмплов: {len(val_dataset)}")

indices = np.linspace(0, len(val_dataset) - 1, N_SAMPLES, dtype=int)
print(f"Выбраны индексы: {indices}")

clean_images = torch.stack([val_dataset[i] for i in indices]).to(DEVICE)
print(f"clean_images shape: {clean_images.shape}")


## Генерация маски трека


In [ ]:
all_files = sorted([
    f for f in os.listdir(SATELLITE_DATA_PATH)
    if os.path.isfile(os.path.join(SATELLITE_DATA_PATH, f))
])
assert all_files, f"Нет файлов в {SATELLITE_DATA_PATH}"
assert 0 <= SATELLITE_FILE_INDEX < len(all_files), (
    f"SATELLITE_FILE_INDEX={SATELLITE_FILE_INDEX} вне диапазона [0, {len(all_files) - 1}]"
)

satellite_file = all_files[SATELLITE_FILE_INDEX]
satellite_full_path = os.path.join(SATELLITE_DATA_PATH, satellite_file)

satellite_data_example = np.load(satellite_full_path)
satellite_data_example = np.pad(
    satellite_data_example[0],
    ((4, 5), (15, 16)),
    mode='constant',
    constant_values=((None, None), (None, None)),
)
satellite_data_mask = torch.from_numpy(np.where(np.isnan(satellite_data_example), 0, 1)).float().to(DEVICE)
print(f"satellite file: {satellite_file}")
print(f"satellite_data_mask shape: {satellite_data_mask.shape}")


## Inference — обусловленный сэмплинг для каждого из сэмплов


In [ ]:
clean = clean_images
mask_batched = satellite_data_mask.unsqueeze(0).unsqueeze(1)
mask_batched = mask_batched.expand(clean_images.shape[0], -1, -1, -1)
observed = clean * mask_batched

predictions = sampler.sample_conditioned(
    mask=mask_batched,
    observed=observed,
    size=IMAGE_SIZE,
    num_timesteps=NUM_TIMESTEPS,
    device=DEVICE,
)

predictions = predictions.cpu()
print(f"predictions shape: {predictions.shape}")


In [ ]:
plt.imshow(satellite_data_mask.cpu())
plt.colorbar()
plt.show()


## Визуализация


In [ ]:
clean_cpu = clean_images.cpu()
pred_cpu = predictions.cpu()
observed_cpu = clean_cpu * satellite_data_mask.cpu()

make_plot(clean_cpu, CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, title="Truth — Concentration (channel 0)")
make_plot(observed_cpu, CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, title="Observed on tracks — Concentration (channel 0)")
make_plot(pred_cpu, CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, title="Prediction — Concentration (channel 0)")


In [ ]:
data_example = np.load(WATER_MASK_SOURCE)
water_mask_initial = np.where(np.isnan(data_example[7, 0, :, :]), 0, 1)
water_mask = np.pad(water_mask_initial, ((4, 5), (15, 16)))
plt.imshow(water_mask)
plt.colorbar()
plt.show()


In [ ]:
make_difference_plot(clean_cpu, pred_cpu, CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, land_mask=water_mask)


## Средний MSE по треку


In [ ]:
truth_dn = channel_denormalize(clean_images.clone().cpu(), CHANNEL_MEAN, CHANNEL_STD)
pred_dn = channel_denormalize(predictions.clone().cpu(), CHANNEL_MEAN, CHANNEL_STD)
mask_cpu = satellite_data_mask.cpu()

diff2 = (truth_dn - pred_dn) ** 2
mask_exp = mask_cpu.unsqueeze(0).unsqueeze(0)

n_pixels = mask_cpu.sum().item()
mse_per_sample = (diff2 * mask_exp).sum(dim=(2, 3)) / n_pixels

print(f"{'Sample':>8}  {'Concentration MSE':>18}  {'Thickness MSE':>14}")
print("-" * 46)
for i in range(N_SAMPLES):
    print(f"  #{indices[i]:>5}  {mse_per_sample[i, 0].item():>18.5f}  {mse_per_sample[i, 1].item():>14.5f}")
print("-" * 46)
print(f"  {'mean':>6}  {mse_per_sample[:, 0].mean().item():>18.5f}  {mse_per_sample[:, 1].mean().item():>14.5f}")
